# 🏠 Ames Housing Dataset — Exploratory Data Analysis
**Proyek Machine Learning: Prediksi Harga Rumah**

Notebook ini mencakup:
1. Pembacaan dataset `train.csv`
2. Handling missing values (imputasi median)
3. Data cleaning — penghapusan outlier menggunakan metode IQR
4. Penyimpanan data bersih ke `train_clean.csv`
5. Visualisasi EDA: Correlation Heatmap & Scatter Plot dengan garis tren

---
## 1. Import Library & Membaca Dataset

In [ ]:
# Import library yang dibutuhkan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Konfigurasi tampilan
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'

print('✅ Library berhasil di-import!')

In [ ]:
# Membaca dataset
df = pd.read_csv('train.csv')

print(f'📊 Dataset berhasil dimuat!')
print(f'   Total baris : {df.shape[0]}')
print(f'   Total kolom : {df.shape[1]}')
print()
df.head()

In [ ]:
# Definisikan fitur numerik yang akan digunakan dan target variabel
features = ['Gr Liv Area', 'Overall Qual', 'Garage Cars', 'Total Bsmt SF', 'Year Built']
target = 'SalePrice'

# Seleksi kolom yang relevan
cols = features + [target]
df_selected = df[cols].copy()

print('📋 Fitur yang dipilih:')
for f in features:
    print(f'   • {f}')
print(f'   🎯 Target: {target}')
print()
df_selected.describe().round(2)

---
## 2. Handling Missing Values (Imputasi Median)

In [ ]:
# Cek missing values sebelum imputasi
missing_before = df_selected.isnull().sum()
print('🔍 Missing Values SEBELUM Imputasi:')
print(missing_before.to_string())
print(f'\n   Total missing: {missing_before.sum()}')

In [ ]:
# Imputasi missing values menggunakan median untuk setiap fitur
for col in cols:
    if df_selected[col].isnull().sum() > 0:
        median_val = df_selected[col].median()
        df_selected[col].fillna(median_val, inplace=True)
        print(f'   ✏️  {col}: {missing_before[col]} nilai kosong diisi dengan median = {median_val}')

# Verifikasi tidak ada lagi missing values
missing_after = df_selected.isnull().sum()
print(f'\n✅ Missing Values SESUDAH Imputasi:')
print(missing_after.to_string())
print(f'\n   Total missing: {missing_after.sum()}')

---
## 3. Data Cleaning — Penghapusan Outlier (Metode IQR)

In [ ]:
# Menghitung IQR pada SalePrice
Q1 = df_selected[target].quantile(0.25)
Q3 = df_selected[target].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f'📐 IQR Analysis pada {target}:')
print(f'   Q1 (25th percentile) : ${Q1:,.0f}')
print(f'   Q3 (75th percentile) : ${Q3:,.0f}')
print(f'   IQR                  : ${IQR:,.0f}')
print(f'   Batas Bawah          : ${lower_bound:,.0f}')
print(f'   Batas Atas           : ${upper_bound:,.0f}')

In [ ]:
# Menghapus outlier ekstrem berdasarkan IQR
rows_before = len(df_selected)

df_clean = df_selected[
    (df_selected[target] >= lower_bound) & 
    (df_selected[target] <= upper_bound)
].copy()

rows_after = len(df_clean)
rows_removed = rows_before - rows_after

print(f'🧹 Hasil Data Cleaning:')
print(f'   Baris sebelum cleaning : {rows_before}')
print(f'   Baris sesudah cleaning : {rows_after}')
print(f'   Outlier dihapus        : {rows_removed} ({rows_removed/rows_before*100:.1f}%)')
print()
df_clean.describe().round(2)

---
## 4. Simpan Data Bersih ke `train_clean.csv`

In [ ]:
# Simpan data yang sudah bersih
df_clean.to_csv('train_clean.csv', index=False)

print('💾 Data bersih berhasil disimpan ke "train_clean.csv"')
print(f'   Ukuran data: {df_clean.shape[0]} baris × {df_clean.shape[1]} kolom')

---
## 5. Visualisasi EDA

### 5a. Correlation Matrix Heatmap

In [ ]:
# ============================
# CORRELATION MATRIX HEATMAP
# ============================

# Hitung matriks korelasi
corr_matrix = df_clean[cols].corr()

# Buat figure
fig, ax = plt.subplots(figsize=(10, 8))

# Custom colormap
cmap = sns.diverging_palette(220, 20, as_cmap=True)

# Buat mask untuk segitiga atas (agar tidak redundan)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Plot heatmap
sns.heatmap(
    corr_matrix,
    mask=mask,
    cmap=cmap,
    vmin=-1, vmax=1,
    center=0,
    annot=True,
    fmt='.2f',
    linewidths=1.5,
    linecolor='white',
    square=True,
    cbar_kws={'shrink': 0.8, 'label': 'Koefisien Korelasi'},
    annot_kws={'size': 12, 'weight': 'bold'},
    ax=ax
)

# Styling
ax.set_title('Correlation Matrix — Fitur Numerik vs SalePrice',
             fontsize=16, fontweight='bold', pad=20)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=11)

plt.tight_layout()

# Simpan grafik sebagai file PNG
fig.savefig('correlation_heatmap.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
print('📊 Grafik disimpan → correlation_heatmap.png')

plt.show()

### 5b. Scatter Plot — Gr Liv Area vs SalePrice (dengan Garis Tren)

In [ ]:
# ============================
# SCATTER PLOT + TREND LINE
# ============================

fig, ax = plt.subplots(figsize=(10, 7))

# Scatter plot dengan warna berdasarkan Overall Qual
scatter = ax.scatter(
    df_clean['Gr Liv Area'],
    df_clean[target],
    c=df_clean['Overall Qual'],
    cmap='viridis',
    alpha=0.6,
    edgecolors='white',
    linewidth=0.5,
    s=50
)

# Garis tren linear (regresi linear)
slope, intercept, r_value, p_value, std_err = stats.linregress(
    df_clean['Gr Liv Area'], df_clean[target]
)
x_line = np.linspace(df_clean['Gr Liv Area'].min(), df_clean['Gr Liv Area'].max(), 100)
y_line = slope * x_line + intercept

ax.plot(x_line, y_line, color='#E74C3C', linewidth=2.5, linestyle='--',
        label=f'Tren Linear (R² = {r_value**2:.3f})')

# Colorbar untuk Overall Quality
cbar = plt.colorbar(scatter, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Overall Quality', fontsize=12)

# Styling
ax.set_xlabel('Luas Bangunan (Gr Liv Area) — sq ft', fontsize=13, fontweight='bold')
ax.set_ylabel('Harga Jual (SalePrice) — USD', fontsize=13, fontweight='bold')
ax.set_title('Scatter Plot — Luas Bangunan vs Harga Jual',
             fontsize=16, fontweight='bold', pad=15)

# Format sumbu Y dalam dollar
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Tambahkan info regresi di dalam plot
textstr = f'y = {slope:.2f}x + {intercept:,.0f}\nR² = {r_value**2:.3f}'
props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8, edgecolor='gray')
ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=props)

ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()

# Simpan grafik sebagai file PNG
fig.savefig('scatter_plot_grliv_vs_saleprice.png', dpi=300, bbox_inches='tight',
            facecolor='white', edgecolor='none')
print('📊 Grafik disimpan → scatter_plot_grliv_vs_saleprice.png')

plt.show()

---
## ✅ Ringkasan

| Tahap | Keterangan |
|-------|------------|
| **Dataset** | Ames Housing — `train.csv` |
| **Fitur** | Gr Liv Area, Overall Qual, Garage Cars, Total Bsmt SF, Year Built |
| **Target** | SalePrice |
| **Missing Values** | Diisi dengan median |
| **Outlier Removal** | Metode IQR pada SalePrice |
| **Output Data** | `train_clean.csv` |
| **Output Grafik** | `correlation_heatmap.png`, `scatter_plot_grliv_vs_saleprice.png` |